# ECG Arrhythmia Detection v2: 1D CNN + RR-interval features (patient-level split)

Classifies single heartbeats as **normal** or **abnormal**. Data is split **by patient**, not by beat, so the test score reflects performance on people the model has never seen.

**v2 changes:** (1) RR-interval timing features (time since previous beat / until next beat) fed to the model next to the waveform, (2) data augmentation + cosine learning-rate schedule to stabilise training, (3) decision threshold chosen on the *validation* patients, and results reported at both 0.5 and that threshold.

**No downloading happens in this notebook.** It reads the MIT-BIH files from a local folder.

### Get the data (one time, ~100 MB)
Run `python download_mitdb.py mitdb` once from a terminal (fast parallel download from PhysioNet's public S3 mirror, resumable). Alternatively, download the ZIP from https://physionet.org/content/mitdb/get-zip/1.0.0/ and unzip it into `mitdb/`. The folder must contain files like `100.dat`, `100.hea`, `100.atr`. Set `DATA_DIR` in the config cell if your folder has a different name.

First time only: `pip install wfdb numpy scikit-learn torch matplotlib`

In [ ]:
import os, glob
import numpy as np
import wfdb
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

## 1. Config: data location and patient-level split

In [ ]:
DATA_DIR = "mitdb"   # <- folder you unzipped the data into (searched recursively)

def find_data_dir(root):
    """Return the folder that actually contains 100.hea (handles the zip's nested folder)."""
    hits = glob.glob(os.path.join(root, "**", "100.hea"), recursive=True)
    if not hits:
        raise FileNotFoundError(
            f"Could not find 100.hea under '{root}'. Download the ZIP from "
            "https://physionet.org/content/mitdb/get-zip/1.0.0/ , unzip it, and set DATA_DIR.")
    return os.path.dirname(hits[0])

DATA_DIR = find_data_dir(DATA_DIR)
print("Using data from:", os.path.abspath(DATA_DIR))

PRE, POST = 90, 90   # samples before / after the R-peak -> 180 samples = 0.5 s at 360 Hz

# Standard inter-patient split (de Chazal et al., 2004). Paced records 102,104,107,217 excluded.
DS1 = [101,106,108,109,112,114,115,116,118,119,122,124,201,203,205,207,208,209,215,220,223,230]
DS2 = [100,103,105,111,113,117,121,123,200,202,210,212,213,214,219,221,222,228,231,232,233,234]
VAL   = [115, 203, 215, 230]                 # held-out TRAIN patients used for model selection
TRAIN = [r for r in DS1 if r not in VAL]
TEST  = DS2                                  # never touched until the very end

BEAT_SYMS = {"N","L","R","e","j","A","a","J","S","V","E","F","/","f","Q"}  # all beat types (for RR timing)
FS = 360
NORMAL   = {"N", "L", "R", "e", "j"}              # -> label 0
ABNORMAL = {"A", "a", "J", "S", "V", "E", "F"}    # -> label 1
# Skipped: '/' (paced), 'f', 'Q' (unclassifiable), and non-beat annotations

missing = [r for r in sorted(set(TRAIN + VAL + TEST))
           if not all(os.path.exists(os.path.join(DATA_DIR, f"{r}.{e}")) for e in ("hea", "dat", "atr"))]
assert not missing, f"Missing files for records: {missing}"
print(f"train={len(TRAIN)} patients, val={len(VAL)}, test={len(TEST)}  (no overlap: "
      f"{not (set(TRAIN) & set(VAL) or set(TRAIN) & set(TEST) or set(VAL) & set(TEST))})")

## 2. Turn records into labelled beats

In [ ]:
def load_record(rec):
    path = os.path.join(DATA_DIR, str(rec))
    sig, fields = wfdb.rdsamp(path)
    ann = wfdb.rdann(path, "atr")
    if sig is None:
        raise ValueError(f"Record {rec} did not return signal data")
    lead = fields["sig_name"].index("MLII") if "MLII" in fields["sig_name"] else 0
    x = sig[:, lead]
    empty = (np.empty((0, PRE + POST), np.float32), np.empty((0, 4), np.float32), np.empty(0, np.int64))
    if ann.symbol is None:
        return empty
    # all beats (any type) with their positions -> RR intervals in seconds
    beats = [(s_, sym) for s_, sym in zip(ann.sample, ann.symbol) if sym in BEAT_SYMS]
    if len(beats) < 3:
        return empty
    pos = np.array([b[0] for b in beats]); rr = np.diff(pos) / FS
    X, R, y = [], [], []
    for j in range(1, len(beats) - 1):
        s_, sym = beats[j]
        if sym in NORMAL: label = 0
        elif sym in ABNORMAL: label = 1
        else: continue
        if s_ - PRE < 0 or s_ + POST > len(x):
            continue
        beat = x[s_ - PRE : s_ + POST]
        beat = (beat - beat.mean()) / (beat.std() + 1e-8)          # per-beat z-score
        pre, post = rr[j - 1], rr[j]
        local = rr[max(0, j - 10):j].mean()                         # recent average RR
        R.append([pre, post, pre / local, post / local])
        X.append(beat); y.append(label)
    if not X:
        return empty
    return np.array(X, np.float32), np.clip(np.array(R, np.float32), 0, 3), np.array(y, np.int64)

def build(records):
    Xs, Rs, ys = [], [], []
    for r in records:
        X, R, y = load_record(r)
        Xs.append(X); Rs.append(R); ys.append(y)
    return np.concatenate(Xs)[:, None, :], np.concatenate(Rs), np.concatenate(ys)   # X:(N,1,180) R:(N,4)

In [ ]:
Xtr, Rtr, ytr = build(TRAIN)
Xva, Rva, yva = build(VAL)
Xte, Rte, yte = build(TEST)
# standardise RR features using TRAIN statistics only
mu, sd = Rtr.mean(0), Rtr.std(0) + 1e-8
Rtr, Rva, Rte = [(R - mu) / sd for R in (Rtr, Rva, Rte)]
for name, X, y in [("train", Xtr, ytr), ("val", Xva, yva), ("test", Xte, yte)]:
    print(f"{name:5s} X={X.shape}  abnormal={y.mean():.2%}")

In [ ]:
# Sanity check: what do normal vs abnormal beats look like?
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)
for k, title in [(0, "Normal beats"), (1, "Abnormal beats")]:
    idx = np.random.choice(np.where(ytr == k)[0], 8, replace=False)
    for i in idx:
        ax[k].plot(Xtr[i, 0], alpha=0.6)
    ax[k].set_title(title); ax[k].set_xlabel("sample (360 Hz)")
plt.tight_layout(); plt.show()

## 3. Model: 1D CNN (waveform) + RR branch (timing)
Three Conv blocks (32/64/128 filters) summarise the beat shape; a small branch encodes 4 RR-interval features; both are concatenated and classified.

In [ ]:
class ECGNet(nn.Module):
    def __init__(self, n_rr=4):
        super().__init__()
        def block(i, o, k):
            return nn.Sequential(nn.Conv1d(i, o, k, padding=k // 2), nn.BatchNorm1d(o),
                                 nn.ReLU(), nn.MaxPool1d(2))
        self.features = nn.Sequential(block(1, 32, 7), block(32, 64, 5), block(64, 128, 3),
                                      nn.AdaptiveAvgPool1d(1), nn.Flatten())
        self.rr = nn.Sequential(nn.Linear(n_rr, 16), nn.ReLU())          # timing branch
        self.head = nn.Sequential(nn.Dropout(0.4), nn.Linear(128 + 16, 64), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(64, 2))
    def forward(self, x, r):
        return self.head(torch.cat([self.features(x), self.rr(r)], 1))

def evaluate(model, loader, device):
    model.eval(); probs, ys = [], []
    with torch.no_grad():
        for xb, rb, yb in loader:
            p = torch.softmax(model(xb.to(device), rb.to(device)), 1)[:, 1].cpu().numpy()
            probs.append(p); ys.append(yb.numpy())
    return np.concatenate(probs), np.concatenate(ys)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
mk = lambda X, R, y, sh: DataLoader(TensorDataset(torch.from_numpy(X), torch.from_numpy(R), torch.from_numpy(y)),
                                    batch_size=256, shuffle=sh)
tr, va, te = mk(Xtr, Rtr, ytr, True), mk(Xva, Rva, yva, False), mk(Xte, Rte, yte, False)

## 4. Train
Weighted cross-entropy handles class imbalance. The validation patients choose the best epoch; the test patients are not used here.

In [ ]:
EPOCHS = 30
model = ECGNet().to(device)
w = torch.tensor([1.0, (ytr == 0).sum() / max((ytr == 1).sum(), 1)], dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

def augment(xb):
    """Small random time shift and noise so the model can't memorise exact patient waveforms."""
    shift = np.random.randint(-6, 7)
    xb = torch.roll(xb, shifts=shift, dims=2)
    return xb + 0.03 * torch.randn_like(xb)

best_auc, best_state, history = 0, None, []
for epoch in range(EPOCHS):
    model.train(); total = 0
    for xb, rb, yb in tr:
        xb, rb, yb = augment(xb).to(device), rb.to(device), yb.to(device)
        opt.zero_grad(); loss = loss_fn(model(xb, rb), yb); loss.backward(); opt.step()
        total += loss.item() * len(xb)
    sched.step()
    p, y = evaluate(model, va, device)
    auc = roc_auc_score(y, p)
    history.append((total / len(ytr), auc))
    print(f"epoch {epoch+1:02d}  train loss {total/len(ytr):.4f}  val AUC {auc:.4f}")
    if auc > best_auc:
        best_auc, best_state = auc, {k: v.clone() for k, v in model.state_dict().items()}

assert best_state is not None
model.load_state_dict(best_state)
torch.save(best_state, "ecg_cnn_v2.pt")
print(f"\nBest val AUC: {best_auc:.4f}  (weights saved to ecg_cnn_v2.pt)")

In [ ]:
h = np.array(history)
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(h[:, 0]); ax[0].set_title("Train loss"); ax[0].set_xlabel("epoch")
ax[1].plot(h[:, 1]); ax[1].set_title("Validation AUC"); ax[1].set_xlabel("epoch")
plt.tight_layout(); plt.show()

## 5. Final evaluation on unseen patients
This is the number to report.

In [ ]:
# 1) Pick the decision threshold on VALIDATION patients (maximise F1 of the abnormal class)
pv, yv = evaluate(model, va, device)
ths = np.linspace(0.05, 0.95, 91)
f1s = [2 * ((pv > t) & (yv == 1)).sum() / (2 * ((pv > t) & (yv == 1)).sum() + ((pv > t) & (yv == 0)).sum() + ((pv <= t) & (yv == 1)).sum() + 1e-9) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print(f"Threshold chosen on validation patients: {thr:.2f}\n")

# 2) Evaluate on unseen TEST patients at both thresholds
p, y = evaluate(model, te, device)
print(f"ROC AUC (test): {roc_auc_score(y, p):.4f}\n")
for t in (0.5, thr):
    pred = (p > t).astype(int)
    cm = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm.ravel()
    print(f"=== threshold {t:.2f} ===")
    print(classification_report(y, pred, target_names=["normal", "abnormal"], digits=4))
    print(f"Sensitivity {tp/(tp+fn):.4f} | Specificity {tn/(tn+fp):.4f} | Precision {tp/(tp+fp+1e-9):.4f}")
    print("Confusion matrix (rows = true, cols = predicted):\n", cm, "\n")

fpr, tpr, _ = roc_curve(y, p)
cm = confusion_matrix(y, (p > thr).astype(int))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(fpr, tpr); ax[0].plot([0, 1], [0, 1], "--", c="gray")
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate"); ax[0].set_title("ROC (test patients)")
ax[1].imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax[1].text(j, i, cm[i, j], ha="center", va="center")
ax[1].set_xticks([0, 1], ["normal", "abnormal"]); ax[1].set_yticks([0, 1], ["normal", "abnormal"])
ax[1].set_xlabel("Predicted"); ax[1].set_ylabel("True"); ax[1].set_title(f"Confusion matrix (threshold {thr:.2f})")
plt.tight_layout(); plt.show()